# Problem Setup
Input: 
- Medical image
- Natural Language question

Output:
- Short textual answer (classification)

In [1]:
import os
import re
import json
from collections import Counter
from PIL import Image
import torch
import torch.nn as nn 
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

### **1. Data Preprocessing**

- Using ImageNet's mean and stddev:
    - [stackoverflow discussion](https://stackoverflow.com/questions/58151507/why-pytorch-officially-use-mean-0-485-0-456-0-406-and-std-0-229-0-224-0-2)
    - [PyTorch documentation](https://docs.pytorch.org/vision/stable/models.html)

In [2]:
# 1.1 Image transformation
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

In [3]:
# 1.2 Text tokenization
# Simple word-level tokenizer
def tokenize(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-z0-9 ]", "", sentence)
    return sentence.split()

# Build vocab
def build_vocab(questions, min_freq=1):
    counter = Counter()
    for q in questions:
        counter.update(tokenize(q))

    vocab = {
        "<pad>": 0, 
        "<unk>": 1
    }

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

# Encode question
def encode_question(question, vocab, max_len=20):
    tokens = tokenize(question)
    ids = [vocab.get(w, vocab["<unk>"]) for w in tokens]
    ids = ids[:max_len]
    return ids + [vocab["<pad>"]] * (max_len - len(ids))

# Answer vocab
def build_answer_vocab(answers):
    unique_answers = set(str(a).lower().strip() for a in answers)
    answer_to_idx = {ans: i for i, ans in enumerate(sorted(unique_answers))}

    return answer_to_idx

In [4]:
# 1.3 Custom dataset class
class VqaRadDataset(Dataset):
    def __init__(self, json_path, image_dir, vocab, answer_to_idx, transform=None):
        with open(json_path) as f:
            self.data = json.load(f)
        
        self.image_dir = image_dir
        self.vocab = vocab
        self.answer_to_idx = answer_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        item = self.data[index]

        # Load image
        image_path = os.path.join(self.image_dir, item['image_name'])
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Encode question
        question = torch.tensor(
            encode_question(item['question'], self.vocab), 
            dtype=torch.long
        )

        answer_text = str(item['answer']).lower().strip()
        answer = torch.tensor(
            self.answer_to_idx[answer_text], 
            dtype=torch.long
        )

        return image, question, answer

In [5]:
with open("VQA_RAD/train.json", "r") as f:
    train_data = json.load(f)

# Extract questions from train data
trans_questions = [item["question"] for item in train_data]
vocab = build_vocab(trans_questions)

print("Vocab size:", len(vocab))

# Extract answers from train data
train_answers = [item["answer"] for item in train_data]
answer_to_idx = build_answer_vocab(train_answers)

print("Number of answer classes:", len(answer_to_idx))
print(answer_to_idx)

Vocab size: 1016
Number of answer classes: 396
{'10-20 minutes': 0, '12': 1, '2': 2, '2.5cm x 1.7cm x 1.6cm': 3, '3.4 cm': 4, '3rd rib': 5, '4': 6, '4th and 5th': 7, '4th ventricle': 8, '5 cm': 9, '5%': 10, '5.6cm focal, predominantly hypodense': 11, '5cm': 12, '7th rib': 13, 'a bit': 14, 'a bullous lesion': 15, 'a catheter': 16, 'abdomen': 17, 'abdomen and pelvis': 18, 'abdominal pain': 19, 'abnormal': 20, 'above the clavicles bilaterally': 21, 'abscess': 22, 'adenopathy': 23, 'adipose tissue': 24, 'adjacent to vertebrae': 25, 'air fluid level': 26, 'air?': 27, 'almost entire right side': 28, 'anterior cerebrum': 29, 'anterior mediastinum': 30, 'anterior surface': 31, 'anterior to the transverse colon': 32, 'aorta enhancement': 33, 'aorta is bright': 34, 'appendicitis': 35, 'appendix': 36, 'ascending colon': 37, 'ascites': 38, 'asymmetric': 39, 'atherosclerotic calcification': 40, 'axial': 41, 'axial plane': 42, 'basal ganglia': 43, 'basal ganglia (caudate and putamen)': 44, 'basal ga

In [6]:
# 1.4 Initialize dataset
DATA_ROOT = "VQA_RAD"
IMAGE_DIR = f"{DATA_ROOT}/images"

train_dataset = VqaRadDataset(
    json_path=f"{DATA_ROOT}/train.json",
    image_dir=IMAGE_DIR,
    vocab=vocab,
    answer_to_idx=answer_to_idx,
    transform=image_transform
)

val_dataset = VqaRadDataset(
    json_path=f"{DATA_ROOT}/val.json",
    image_dir=IMAGE_DIR,
    vocab=vocab,
    answer_to_idx=answer_to_idx,
    transform=image_transform
)

test_dataset = VqaRadDataset(
    json_path=f"{DATA_ROOT}/test.json",
    image_dir=IMAGE_DIR,
    vocab=vocab,
    answer_to_idx=answer_to_idx,
    transform=image_transform
)

In [7]:
# 1.5 Initialize DataLoader
BATCH_SIZE = 32

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

In [8]:
# Verifying batch
image, question, answer = next(iter(train_loader))

print(image.shape)
print(question.shape)
print(answer.shape)

/Users/alyani/Desktop/grad school/WOA7015/Final Assessment/FINAL_ASSESSMENT_WOA7015/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


torch.Size([32, 3, 224, 224])
torch.Size([32, 20])
torch.Size([32])


### **2. Model Architecture**
- CNN (image) + LSTM (text) -> Fusion (Classifier)

In [9]:
# 2.1 Image encoder (CNN)
class ImageEncoder(nn.Module):

    def __init__(self, output_dim=512):
        super().__init__()
        resnet = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        self.image_fc = nn.Linear(512, output_dim)

    def forward(self, x):
        x = self.cnn(x).squeeze()
        return self.image_fc(x)

In [10]:
# 2.2 Text input encoder (LSTM)
class TextInputEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, hidden_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        emb = self.embedding(x)
        _, (h, _) = self.lstm(emb)
        return h[-1]

In [11]:
# 2.3 VQA Model (Fusion)
class VQAModel(nn.Module):
    def __init__(self, vocab_size, num_answers):
        super().__init__()
        self.image_encoder = ImageEncoder()
        self.text_input_encoder = TextInputEncoder(vocab_size)

        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_answers)
        )

    def forward(self, image, question):
        img_features = self.image_encoder(image)
        ques_feature = self.text_input_encoder(question)
        fused = torch.cat([img_features, ques_feature], dim=1)
        return self.classifier(fused)

### **3. Training Pipeline**

In [12]:
model = VQAModel(vocab_size=len(vocab), num_answers=len(answer_to_idx))
# model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, questions, answers in train_loader:
        #images, questions, answers = images.to(device), questions.to(device), answers.to(device)

        optimizer.zero_grad()
        outputs = model(images, questions)
        loss = criterion(outputs, answers)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

/Users/alyani/Desktop/grad school/WOA7015/Final Assessment/FINAL_ASSESSMENT_WOA7015/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/alyani/Desktop/grad school/WOA7015/Final Assessment/FINAL_ASSESSMENT_WOA7015/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
5.3%

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/alyani/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


Epoch 1, Loss: 4.5557
Epoch 2, Loss: 3.6971
Epoch 3, Loss: 3.2721


### **4. Model Evaluation**

In [ ]:
def eval(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, questions, answers in loader:
            outputs = model(images, questions)
            preds = outputs.argmax(dim=1)
            correct += (preds == answers).sum().item()
            total += answers.size(0)

    return correct / total